# 04 — Análise de Vendas

Notebook dedicado à análise analítica das vendas consolidadas no Data Warehouse Hive.

São analisados faturamento, pedidos, itens, ticket médio, clientes, produtos e categorias, utilizando a tabela fato `fact_sales` e as dimensões `dim_customer`, `dim_product` e `dim_category`.

**Camada:** `ecommerce` / `ecommerce_analytics`


## 1. Configuração

Define a conexão com o Hive e os parâmetros utilizados pelas consultas analíticas.


In [ ]:
import os
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

HIVE_BIN = os.getenv("HIVE_BIN", "hive")
DATABASE = "ecommerce"
ANALYTICS_DATABASE = "ecommerce_analytics"

print(f"Hive: {HIVE_BIN}")
print(f"Banco de fatos: {DATABASE}")
print(f"Banco analítico: {ANALYTICS_DATABASE}")


## 2. Função de consulta Hive

In [ ]:
def hive_query(sql):
    result = subprocess.run(
        [HIVE_BIN, "-e", sql],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            result.stderr.strip() or "Falha na execução da consulta Hive"
        )

    return result.stdout.strip()


hive_available = True

try:
    print(hive_query("SHOW DATABASES;"))
except Exception as exc:
    hive_available = False
    print(f"Hive indisponível no momento: {exc}")


## 3. Visão geral das vendas

A primeira análise apresenta os principais indicadores consolidados da tabela `fact_sales`.


In [ ]:
if hive_available:
    query = """
    SELECT
        COUNT(*) AS total_sales,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT customer_id) AS unique_customers,
        SUM(quantity) AS total_items,
        ROUND(SUM(total_amount), 2) AS gross_revenue,
        ROUND(AVG(total_amount), 2) AS average_order_value
    FROM ecommerce.fact_sales
    WHERE status = 'COMPLETED'
    """

    overview = hive_query(query)
    print(overview)
else:
    print("Execute esta análise com o Hive disponível.")


## 4. Vendas por dia

Analisa a evolução diária do faturamento, pedidos e quantidade de itens vendidos.


In [ ]:
daily_sales = pd.DataFrame()

if hive_available:
    query = """
    SELECT
        event_date,
        COUNT(*) AS total_sales,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(quantity) AS total_items,
        ROUND(SUM(total_amount), 2) AS revenue
    FROM ecommerce.fact_sales
    WHERE status = 'COMPLETED'
    GROUP BY event_date
    ORDER BY event_date
    """

    result = hive_query(query)

    if result:
        rows = [line.split("\t") for line in result.splitlines()]
        daily_sales = pd.DataFrame(
            rows,
            columns=[
                "event_date",
                "total_sales",
                "total_orders",
                "total_items",
                "revenue",
            ],
        )

        daily_sales["event_date"] = pd.to_datetime(daily_sales["event_date"])
        for column in ["total_sales", "total_orders", "total_items", "revenue"]:
            daily_sales[column] = pd.to_numeric(
                daily_sales[column],
                errors="coerce",
            )

        display(daily_sales)
else:
    print("Sem conexão com o Hive.")


## 5. Evolução do faturamento

Visualiza a evolução temporal da receita consolidada.


In [ ]:
if not daily_sales.empty:
    plt.figure(figsize=(10, 5))
    plt.plot(
        daily_sales["event_date"],
        daily_sales["revenue"],
        marker="o",
    )
    plt.title("Evolução diária do faturamento")
    plt.xlabel("Data")
    plt.ylabel("Faturamento")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Não há dados suficientes para gerar o gráfico.")


## 6. Produtos com maior faturamento

Relaciona a tabela fato de vendas com as dimensões de produtos e categorias.


In [ ]:
if hive_available:
    query = """
    SELECT
        s.product_id,
        p.name AS product_name,
        p.category_id,
        c.name AS category_name,
        SUM(s.quantity) AS total_quantity,
        COUNT(DISTINCT s.order_id) AS total_orders,
        ROUND(SUM(s.total_amount), 2) AS total_revenue
    FROM ecommerce.fact_sales s
    LEFT JOIN ecommerce.dim_product p
        ON s.product_id = p.product_id
    LEFT JOIN ecommerce.dim_category c
        ON p.category_id = c.category_id
    WHERE s.status = 'COMPLETED'
    GROUP BY
        s.product_id,
        p.name,
        p.category_id,
        c.name
    ORDER BY total_revenue DESC
    LIMIT 20
    """

    result = hive_query(query)

    if result:
        rows = [line.split("\t") for line in result.splitlines()]
        products = pd.DataFrame(
            rows,
            columns=[
                "product_id",
                "product_name",
                "category_id",
                "category_name",
                "total_quantity",
                "total_orders",
                "total_revenue",
            ],
        )

        for column in ["total_quantity", "total_orders", "total_revenue"]:
            products[column] = pd.to_numeric(
                products[column],
                errors="coerce",
            )

        display(products)
    else:
        print("Nenhuma venda encontrada.")
else:
    print("Hive indisponível.")


## 7. Faturamento por categoria

In [ ]:
if hive_available:
    query = """
    SELECT
        c.category_id,
        c.name AS category_name,
        SUM(s.quantity) AS total_items,
        COUNT(DISTINCT s.order_id) AS total_orders,
        ROUND(SUM(s.total_amount), 2) AS total_revenue
    FROM ecommerce.fact_sales s
    LEFT JOIN ecommerce.dim_product p
        ON s.product_id = p.product_id
    LEFT JOIN ecommerce.dim_category c
        ON p.category_id = c.category_id
    WHERE s.status = 'COMPLETED'
    GROUP BY
        c.category_id,
        c.name
    ORDER BY total_revenue DESC
    """

    result = hive_query(query)

    if result:
        rows = [line.split("\t") for line in result.splitlines()]
        category_sales = pd.DataFrame(
            rows,
            columns=[
                "category_id",
                "category_name",
                "total_items",
                "total_orders",
                "total_revenue",
            ],
        )

        for column in ["total_items", "total_orders", "total_revenue"]:
            category_sales[column] = pd.to_numeric(
                category_sales[column],
                errors="coerce",
            )

        display(category_sales)
else:
    print("Hive indisponível.")


## 8. Clientes com maior faturamento

Identifica os clientes que apresentam maior valor acumulado de compras.


In [ ]:
if hive_available:
    query = """
    SELECT
        s.customer_id,
        c.name AS customer_name,
        c.segment,
        COUNT(DISTINCT s.order_id) AS total_orders,
        SUM(s.quantity) AS total_items,
        ROUND(SUM(s.total_amount), 2) AS total_spent,
        ROUND(AVG(s.total_amount), 2) AS average_order_value
    FROM ecommerce.fact_sales s
    LEFT JOIN ecommerce.dim_customer c
        ON s.customer_id = c.customer_id
    WHERE s.status = 'COMPLETED'
    GROUP BY
        s.customer_id,
        c.name,
        c.segment
    ORDER BY total_spent DESC
    LIMIT 20
    """

    result = hive_query(query)

    if result:
        rows = [line.split("\t") for line in result.splitlines()]
        customer_sales = pd.DataFrame(
            rows,
            columns=[
                "customer_id",
                "customer_name",
                "segment",
                "total_orders",
                "total_items",
                "total_spent",
                "average_order_value",
            ],
        )

        for column in [
            "total_orders",
            "total_items",
            "total_spent",
            "average_order_value",
        ]:
            customer_sales[column] = pd.to_numeric(
                customer_sales[column],
                errors="coerce",
            )

        display(customer_sales)
else:
    print("Hive indisponível.")


## 9. Ticket médio por período

Calcula o valor médio das vendas concluídas em cada data.


In [ ]:
if hive_available:
    query = """
    SELECT
        event_date,
        ROUND(AVG(total_amount), 2) AS average_order_value,
        COUNT(DISTINCT order_id) AS total_orders
    FROM ecommerce.fact_sales
    WHERE status = 'COMPLETED'
    GROUP BY event_date
    ORDER BY event_date
    """

    result = hive_query(query)

    if result:
        rows = [line.split("\t") for line in result.splitlines()]
        ticket_daily = pd.DataFrame(
            rows,
            columns=[
                "event_date",
                "average_order_value",
                "total_orders",
            ],
        )

        ticket_daily["event_date"] = pd.to_datetime(ticket_daily["event_date"])
        ticket_daily["average_order_value"] = pd.to_numeric(
            ticket_daily["average_order_value"],
            errors="coerce",
        )
        ticket_daily["total_orders"] = pd.to_numeric(
            ticket_daily["total_orders"],
            errors="coerce",
        )

        display(ticket_daily)
else:
    print("Hive indisponível.")


## 10. Produtos mais vendidos

Ordena os produtos pela quantidade total de unidades comercializadas.


In [ ]:
if hive_available:
    query = """
    SELECT
        s.product_id,
        p.name AS product_name,
        SUM(s.quantity) AS total_quantity,
        ROUND(SUM(s.total_amount), 2) AS total_revenue
    FROM ecommerce.fact_sales s
    LEFT JOIN ecommerce.dim_product p
        ON s.product_id = p.product_id
    WHERE s.status = 'COMPLETED'
    GROUP BY
        s.product_id,
        p.name
    ORDER BY total_quantity DESC
    LIMIT 20
    """

    result = hive_query(query)

    if result:
        rows = [line.split("\t") for line in result.splitlines()]
        top_products = pd.DataFrame(
            rows,
            columns=[
                "product_id",
                "product_name",
                "total_quantity",
                "total_revenue",
            ],
        )

        top_products["total_quantity"] = pd.to_numeric(
            top_products["total_quantity"],
            errors="coerce",
        )
        top_products["total_revenue"] = pd.to_numeric(
            top_products["total_revenue"],
            errors="coerce",
        )

        display(top_products)
else:
    print("Hive indisponível.")


## 11. Validação da view `vw_sales_daily`

A consulta abaixo valida a view analítica criada no Hive para consolidação das métricas diárias de vendas.


In [ ]:
if hive_available:
    try:
        result = hive_query(
            "SELECT * FROM ecommerce_analytics.vw_sales_daily LIMIT 20;"
        )
        print(result or "View sem registros.")
    except Exception as exc:
        print(f"Não foi possível consultar a view: {exc}")
else:
    print("Hive indisponível.")


## 12. Principais indicadores

As consultas anteriores permitem obter os principais KPIs comerciais:

- faturamento bruto;
- quantidade de pedidos;
- quantidade de itens;
- clientes únicos;
- ticket médio;
- produtos com maior volume;
- produtos com maior faturamento;
- categorias com maior receita;
- clientes com maior valor acumulado.

Esses indicadores demonstram o uso do Hive como camada consolidada para análise das vendas processadas pela pipeline Big Data.


## 13. Conclusão

A análise de vendas utiliza a tabela fato `fact_sales` em conjunto com as dimensões de clientes, produtos e categorias. As agregações e joins permitem transformar os eventos de pedidos em indicadores comerciais consolidados.

O notebook também demonstra o consumo das views da camada `ecommerce_analytics`, conectando o processamento batch e o Data Warehouse à etapa de análise.
